In [60]:
import pandas as pd
import numpy as np


In [61]:
df = pd.read_csv("stock_data.csv")
df.head()

,Unnamed: 0,open,open.1,open.2,open.3,open.4,open.5,open.6,open.7,open.8,...,volume.14,volume.15,volume.16,volume.17,volume.18,volume.19,volume.20,volume.21,volume.22,volume.23
0,symbol,AAPL,ABBV,AMZN,AVGO,BAC,BRK-B,COST,CVX,GOOG,...,META,MSFT,MU,NFLX,NVDA,ORCL,TSLA,V,WMT,XOM
1,date,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-02,27.84749984741211,65.44000244140625,15.628999710083008,10.093000411987305,17.989999771118164,151.5,141.8699951171875,111.62999725341797,26.37807846069336,...,18177500.0,27913900.0,15138200.0,134750000.0,113680000.0,15070200.0,71466000.0,8389600.0,13505400.0,10220400.0
3,2015-01-05,27.072500228881836,65.5,15.350500106811523,10.006999969482422,17.790000915527344,148.80999755859375,141.69000244140625,110.95999908447266,26.091365814208984,...,26452200.0,39673900.0,23706600.0,181650000.0,197952000.0,18369400.0,80527500.0,12751200.0,20937000.0,18502400.0
4,2015-01-06,26.635000228881836,65.62000274658203,15.112000465393066,9.895000457763672,17.420000076293945,147.63999938964844,140.61000061035156,107.87000274658203,25.67949676513672,...,27399300.0,36447900.0,40092000.0,160377000.0,197764000.0,19229500.0,93928500.0,11070000.0,24615300.0,16670700.0


In [62]:
import pandas as pd

raw = pd.read_csv("stock_data.csv")
symbols = raw.iloc[0]
df = raw.iloc[2:].copy()

df = df.rename(columns={"Unnamed: 0": "date"})
df["date"] = pd.to_datetime(df["date"])
data_cols = df.columns[1:]

new_cols = []
for col in data_cols:
    field = col.split(".")[0]      # open/high/low/adjclose/volume
    company = symbols[col]         # AAPL, XOM, ...
    new_cols.append((company, field))

df_data = df[data_cols].copy()
df_data.columns = pd.MultiIndex.from_tuples(new_cols, names=["company", "field"])


df_data.index = df["date"]

result = df_data.stack(level="company")
result.index = result.index.set_names(["date", "company"])
result = result.swaplevel().sort_index()

print(result.head())

field                             open                high  \
company date                                                 
AAPL    2015-01-02   27.84749984741211  27.860000610351562   
        2015-01-05  27.072500228881836  27.162500381469727   
        2015-01-06  26.635000228881836  26.857500076293945   
        2015-01-07  26.799999237060547  27.049999237060547   
        2015-01-08    27.3075008392334  28.037500381469727   

field                              low            adjclose       volume  
company date                                                             
AAPL    2015-01-02  26.837499618530273  24.214895248413086  212818400.0  
        2015-01-05  26.352500915527344  23.532724380493164  257142000.0  
        2015-01-06  26.157499313354492  23.534931182861328  263188400.0  
        2015-01-07  26.674999237060547   23.86494255065918  160423600.0  
        2015-01-08  27.174999237060547   24.78189468383789  237458000.0  


/tmp/ipykernel_418/2214247361.py:23: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  result = df_data.stack(level="company")


In [63]:
result

field                             open                high  \
company date                                                 
AAPL    2015-01-02   27.84749984741211  27.860000610351562   
        2015-01-05  27.072500228881836  27.162500381469727   
        2015-01-06  26.635000228881836  26.857500076293945   
        2015-01-07  26.799999237060547  27.049999237060547   
        2015-01-08    27.3075008392334  28.037500381469727   
...                                ...                 ...   
XOM     2025-12-23  118.47000122070312  120.02999877929688   
        2025-12-24  119.33000183105469  120.05000305175781   
        2025-12-26  118.88999938964844  119.55000305175781   
        2025-12-29   120.1500015258789  121.30000305175781   
        2025-12-30   121.0999984741211  121.80000305175781   

field                              low            adjclose       volume  
company date                                                             
AAPL    2015-01-02  26.837499618530273  24.214895248413086  212818400.0  
        2015-01-05  26.352500915527344  23.532724380493164  257142000.0  
        2015-01-06  26.157499313354492  23.534931182861328  263188400.0  
        2015-01-07  26.674999237060547   23.86494255065918  160423600.0  
        2015-01-08  27.174999237060547   24.78189468383789  237458000.0  
...                                ...                 ...          ...  
XOM     2025-12-23  118.31999969482422  118.62928771972656   12567600.0  
        2025-12-24  119.12000274658203  118.43061828613281    6137400.0  
        2025-12-26  118.52999877929688  118.32134246826172    8066100.0  
        2025-12-29   119.4000015258789  119.73194122314453   14782500.0  
        2025-12-30  120.62999725341797  120.18889617919922   11150500.0  

[66360 rows x 5 columns]

In [64]:

df = result.copy()

df = df.sort_index()

In [65]:
# Alpha#1: (-1 * Ts_Rank(rank(low), 9)) 

low_rank = df['low'].groupby('date').rank(pct=True)

alpha1 = low_rank.groupby('company').rolling(9).apply(
    lambda x: x.rank(pct=True).iloc[-1]
).reset_index(level=0, drop=True)

df['alpha1'] = -1 * alpha1

In [66]:
# Alpha#2: (-1 * correlation(open, volume, 10)) 

alpha2 = df.groupby('company').apply(
    lambda x: x['open'].rolling(10).corr(x['volume'])
)

df['alpha2'] = -1 * alpha2.reset_index(level=0, drop=True)

In [67]:
# Alpha#3: (-1 * rank(covariance(rank(close), rank(volume), 5)))
close_rank = df['adjclose'].groupby(level='date').rank(pct=True)
volume_rank = df['volume'].groupby(level='date').rank(pct=True)


cov = (
    pd.DataFrame({'close_rank': close_rank, 'volume_rank': volume_rank})
    .groupby(level='company')
    .apply(lambda x: x['close_rank'].rolling(5).cov(x['volume_rank']))
)

cov.index = cov.index.droplevel(0)


alpha3 = cov.groupby(level='date').rank(pct=True)


df['alpha3'] = -alpha3

In [68]:
# Alpha#4: rank((-1 * ((1 - (open / close))^1)))
df['open'] = pd.to_numeric(df['open'], errors='coerce')
df['adjclose'] = pd.to_numeric(df['adjclose'], errors='coerce')


alpha4_raw = -(1 - df['open'] / df['adjclose'])
df['alpha4'] = alpha4_raw.groupby(level='date').rank(pct=True)

In [55]:
# Alpha#5 
price_cols = ['open', 'high', 'low', 'adjclose', 'volume']
for col in price_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

num = -((df['low'] - df['adjclose']) * (df['open'] ** 5))
den = (df['low'] - df['high']) * (df['adjclose'] ** 5)

alpha54 = num / den
alpha54 = alpha54.replace([np.inf, -np.inf], np.nan)

df['alpha5'] = alpha54

In [56]:
alpha101 = (df['adjclose'] - df['open']) / ((df['high'] - df['low']) + 0.001)

df['alpha6'] = alpha101

In [58]:
df

field                     open        high         low    adjclose  \
company date                                                         
AAPL    2015-01-02   27.847500   27.860001   26.837500   24.214895   
        2015-01-05   27.072500   27.162500   26.352501   23.532724   
        2015-01-06   26.635000   26.857500   26.157499   23.534931   
        2015-01-07   26.799999   27.049999   26.674999   23.864943   
        2015-01-08   27.307501   28.037500   27.174999   24.781895   
...                        ...         ...         ...         ...   
XOM     2025-12-23  118.470001  120.029999  118.320000  118.629288   
        2025-12-24  119.330002  120.050003  119.120003  118.430618   
        2025-12-26  118.889999  119.550003  118.529999  118.321342   
        2025-12-29  120.150002  121.300003  119.400002  119.731941   
        2025-12-30  121.099998  121.800003  120.629997  120.188896   

field                    volume    alpha1    alpha2    alpha3    alpha4  \
company date                                                              
AAPL    2015-01-02  212818400.0       NaN       NaN       NaN  0.500000   
        2015-01-05  257142000.0       NaN       NaN       NaN  0.500000   
        2015-01-06  263188400.0       NaN       NaN       NaN  0.500000   
        2015-01-07  160423600.0       NaN       NaN       NaN  0.500000   
        2015-01-08  237458000.0       NaN       NaN -0.958333  0.500000   
...                         ...       ...       ...       ...       ...   
XOM     2025-12-23   12567600.0 -0.666667  0.307551 -0.416667  0.541667   
        2025-12-24    6137400.0 -0.611111  0.419496 -0.479167  0.958333   
        2025-12-26    8066100.0 -0.611111  0.459124 -0.500000  0.666667   
        2025-12-29   14782500.0 -0.611111  0.432267 -0.375000  0.625000   
        2025-12-30   11150500.0 -0.555556  0.447712 -0.583333  0.791667   

field                  alpha5    alpha6  
company date                             
AAPL    2015-01-02   5.159257 -3.549195  
        2015-01-05   7.014727 -4.364708  
        2015-01-06   6.955480 -4.422348  
        2015-01-07  13.383070 -7.806002  
        2015-01-08   4.507536 -2.924844  
...                       ...       ...  
XOM     2025-12-23  -0.179659  0.093096  
        2025-12-24   0.769851 -0.966040  
        2025-12-26   0.209527 -0.556958  
        2025-12-29  -0.177776 -0.219916  
        2025-12-30   0.391516 -0.778051  

[66360 rows x 11 columns]

In [59]:
df.to_csv("alpha_factors.csv")